In [1]:
import torchvision
import torch
import torch.nn as nn
import torchvision.transforms as transforms

# 1. Load and Preprocess Data
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

trainset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=4, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=4, shuffle=False, num_workers=2)


In [2]:
class Lenet5(nn.Module):
  def __init__(self):
    super(Lenet5, self).__init__()
    #in (3x32x32)
    self.conv1 = nn.Conv2d(in_channels = 3, out_channels = 6, kernel_size = 5, stride = 1) # đầu ra: (32, 6, 28, 28), batch_size = 32
    self.relu1 = nn.ReLU()
    #in (32, 6, 28, 28)
    self.pool1 = nn.AvgPool2d(kernel_size = 2, stride = 2) # đầu ra: (32, 6, 14, 14)

    self.conv2 = nn.Conv2d(in_channels = 6, out_channels = 16, kernel_size = 5, stride = 1) # đầu ra: (32, 16, 10, 10)
    self.relu2 = nn.ReLU()
    self.pool2 = nn.AvgPool2d(kernel_size = 2, stride = 2) # đầu ra: (32, 16, 5, 5)

    self.flatten = nn.Flatten() # đầu ra: (32, 16 * 5 * 5)

    self.fc1 = nn.Linear(in_features = 16 * 5 * 5, out_features = 120) # đầu ra: (32, 120)
    self.relu3 = nn.ReLU()

    self.fc2 = nn.Linear(in_features = 120, out_features = 100) # đầu ra: (32, 100)
    # self.relu4 = nn.ReLU()

    # self.fc3 = nn.Linear(in_features = 84, out_features = 10) # đầu ra (32, 10)

  def forward(self, x):
    x = self.conv1(x)
    x = self.relu1(x)
    x = self.pool1(x)

    x = self.conv2(x)
    x = self.relu2(x)
    x = self.pool2(x)

    x = self.flatten(x)

    x = self.fc1(x)
    x = self.relu3(x)

    x = self.fc2(x)
    # x = self.relu4(x)

    # x = self.fc3(x)

    return x

In [3]:
def train_and_evaluate(model, trainloader, testloader, criterion, optimizer, num_epochs = 10):
  for epoch in range(num_epochs):
    model.train()
    for i, data in enumerate(trainloader):
      inputs, labels = data
      optimizer.zero_grad()
      outputs = model(inputs)
      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
      for data in testloader:
        images, labels = data
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    accuracy = 100 * correct / total
    print(f"Epoch {epoch + 1}, accuracy: {accuracy:.2f}%")

In [4]:
criteria = nn.CrossEntropyLoss()
model = Lenet5()
optimizer = torch.optim.SGD(model.parameters(), lr = 0.001)
train_and_evaluate(model, trainloader, testloader, criteria, optimizer)

Epoch 1, accuracy: 3.02%
Epoch 2, accuracy: 5.92%
Epoch 3, accuracy: 10.49%
Epoch 4, accuracy: 12.88%
Epoch 5, accuracy: 15.45%
Epoch 6, accuracy: 17.17%
Epoch 7, accuracy: 17.51%
Epoch 8, accuracy: 19.68%
Epoch 9, accuracy: 20.53%
Epoch 10, accuracy: 20.83%
